# Semana 01 · Tracking MLOps con MLflow en Databricks

En esta práctica vas a entrenar modelos **muy sencillos** para centrarte en el ciclo MLOps: datos, parámetros, métricas, artefactos, comparación y una decisión reproducible. El objetivo no es ganar una competición de machine learning, sino dejar evidencia clara en MLflow.


## Mapa y criterios de terminación

Al acabar debes poder enseñar:

1. un experimento de MLflow con varios runs comparables;
2. tags que identifiquen tu alias, lote y propósito docente;
3. parámetros y métricas de validación/test;
4. un modelo ganador guardado como artefacto;
5. un pequeño smoke test de inferencia.


## 0. Dependencias

En Databricks, ejecuta `%pip` al principio. Si el entorno reinicia Python, continúa desde la celda de imports.


In [ ]:
%pip install -q --upgrade "mlflow[databricks]" "scikit-learn" "pandas" "matplotlib"

## 1. Configuración e identidad del lote

Usa rutas de Databricks (`/dbfs/...`) para artefactos pequeños que quieras inspeccionar fuera de MLflow.


In [ ]:
import json
import time
import uuid
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
STUDENT_ALIAS = "TODO-tu-alias"  # TODO 1: decide un alias estable; importa porque lo usarás para filtrar tus runs; inspecciona tags de MLflow; verifica que aparece en search_runs.
BATCH_ID = f"s01-{uuid.uuid4().hex[:8]}"
EXPERIMENT_NAME = "/Shared/muiaap-operacion-modelos/semana01-tracking"
ARTIFACT_DIR = Path("/dbfs/tmp/muiaap_s01_tracking")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_experiment(EXPERIMENT_NAME)
print({"batch_id": BATCH_ID, "alias": STUDENT_ALIAS, "artifact_dir": str(ARTIFACT_DIR)})

## 2. Datos pequeños y contrato mínimo

Usaremos `load_wine` de scikit-learn: no descarga datos, es rápido y suficiente para practicar tracking.


In [ ]:
wine = load_wine(as_frame=True)
data = wine.frame.rename(columns={"target": "quality_class"})
FEATURES = list(wine.feature_names)
TARGET = "quality_class"

assert TARGET in data.columns
assert data[FEATURES].isna().sum().sum() == 0
assert data[TARGET].nunique() == 3

display(data.head())
print(data.shape, data[TARGET].value_counts().sort_index().to_dict())

## 3. Separar train/validación/test

La celda ya trae una partición sencilla para que la práctica sea ejecutable en Databricks.

**TODO 2:** revisa si el reparto 70/15/15 mantiene todas las clases en validación y test. Decide si cambiarías el porcentaje de test en un proyecto real, porque la evaluación debe ser comparable y suficientemente estable. Inspecciona `train_test_split` y el parámetro `stratify`. Verifica con los `assert` y los conteos por clase.


In [ ]:
X = data[FEATURES]
y = data[TARGET]

# Partición base 70/15/15. TODO 2: revisa los tamaños y conteos de clase antes de aceptar este split.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

assert len(X_train) + len(X_valid) + len(X_test) == len(data)
print({"train": len(X_train), "valid": len(X_valid), "test": len(X_test)})
print("valid classes:", y_valid.value_counts().sort_index().to_dict())
print("test classes:", y_test.value_counts().sort_index().to_dict())


## 4. Candidatos simples

Mantén pocos candidatos para que el foco esté en MLflow. Un baseline, una regresión logística y un bosque pequeño son suficientes.

**TODO 3:** revisa los parámetros y decide si cambiarías uno. Importa porque los parámetros deben explicar diferencias entre runs. Inspecciona la API del estimador elegido y verifica que cada candidato tiene nombre y modelo.


In [ ]:
CANDIDATES = [
    {
        "name": "baseline_most_frequent",
        "model": DummyClassifier(strategy="most_frequent"),
        "params": {"family": "baseline", "strategy": "most_frequent"},
    },
    {
        "name": "logistic_regression",
        "model": Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=500, random_state=SEED)),
        ]),
        "params": {"family": "linear", "max_iter": 500},
    },
    {
        "name": "random_forest_small",
        "model": RandomForestClassifier(n_estimators=60, max_depth=4, random_state=SEED),
        "params": {"family": "trees", "n_estimators": 60, "max_depth": 4},
    },
]

assert all("name" in c and "model" in c for c in CANDIDATES)
[c["name"] for c in CANDIDATES]

## 5. Registrar runs comparables

Cada run debe contar la misma historia: identidad, parámetros, métrica de validación y artefactos mínimos.

**TODO 4:** completa o revisa los tags. Importa porque después filtrarás sólo tus runs. Inspecciona `mlflow.set_tags`, `mlflow.log_params`, `mlflow.log_metrics` y `mlflow.sklearn.log_model`. Verifica que `run_infos` contiene un `model_uri` por candidato.


In [ ]:
run_infos = []

for candidate in CANDIDATES:
    with mlflow.start_run(run_name=f"{BATCH_ID}-{candidate['name']}") as run:
        mlflow.set_tags({
            "course": "operacion-modelos",
            "week": "01",
            "student.alias": STUDENT_ALIAS,
            "batch_id": BATCH_ID,
            "purpose": "guided-mlflow-tracking",
        })
        mlflow.log_params(candidate["params"])
        mlflow.log_param("feature_count", len(FEATURES))
        mlflow.log_param("train_rows", len(X_train))
        mlflow.log_param("validation_rows", len(X_valid))

        start = time.perf_counter()
        model = candidate["model"]
        model.fit(X_train, y_train)
        valid_pred = model.predict(X_valid)
        latency_ms = (time.perf_counter() - start) * 1000

        metrics = {
            "validation.accuracy": accuracy_score(y_valid, valid_pred),
            "validation.f1_macro": f1_score(y_valid, valid_pred, average="macro"),
            "train_latency_ms": latency_ms,
        }
        mlflow.log_metrics(metrics)

        report = classification_report(y_valid, valid_pred, output_dict=True)
        report_path = ARTIFACT_DIR / f"{candidate['name']}_validation_report.json"
        report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
        mlflow.log_artifact(str(report_path), artifact_path="validation")
        model_info = mlflow.sklearn.log_model(model, name="model", input_example=X_train.head(3))

        run_infos.append({
            "run_id": run.info.run_id,
            "name": candidate["name"],
            "model_uri": model_info.model_uri,
            **metrics,
        })

runs_df = pd.DataFrame(run_infos).sort_values("validation.f1_macro", ascending=False)
display(runs_df)
assert len(runs_df) == len(CANDIDATES)
assert runs_df["model_uri"].notna().all()

## 6. Buscar tus runs y elegir ganador sin mirar test

**TODO 5:** filtra por `student.alias` y `batch_id`. Importa porque en un workspace compartido hay runs de otras personas. Inspecciona `mlflow.search_runs` y verifica que no aparecen runs de otro lote.


In [ ]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
filter_string = f"tags.student.alias = '{STUDENT_ALIAS}' and tags.batch_id = '{BATCH_ID}'"
searched = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=filter_string,
    order_by=["metrics.validation.f1_macro DESC"],
)

display(searched[["run_id", "tags.mlflow.runName", "metrics.validation.f1_macro", "metrics.validation.accuracy"]])
assert set(searched["tags.batch_id"]) == {BATCH_ID}

best_row = searched.iloc[0]
BEST_RUN_ID = best_row["run_id"]
BEST_MODEL_URI = f"runs:/{BEST_RUN_ID}/model"
print({"best_run_id": BEST_RUN_ID, "best_model_uri": BEST_MODEL_URI})

## 7. Evaluar test una sola vez

Test se usa después de elegir. Esto evita optimizar contra la evidencia final.

**TODO 6:** registra métricas de test en el run ganador. Importa porque la decisión queda trazada en el mismo run. Inspecciona `mlflow.start_run(run_id=...)` y verifica que las métricas `test.*` aparecen en MLflow.


In [ ]:
best_model = mlflow.sklearn.load_model(BEST_MODEL_URI)
test_pred = best_model.predict(X_test)
test_metrics = {
    "test.accuracy": accuracy_score(y_test, test_pred),
    "test.f1_macro": f1_score(y_test, test_pred, average="macro"),
}

with mlflow.start_run(run_id=BEST_RUN_ID):
    mlflow.log_metrics(test_metrics)
    mlflow.set_tags({"selection.stage": "tested", "selection.reason": "best_validation_f1_macro"})

print(test_metrics)

## 8. Smoke test de inferencia

Comprueba que un consumidor puede cargar el modelo por URI y predecir pocas filas. No levantamos API local ni usamos CLI: eso llegará en semanas posteriores.


In [ ]:
sample = X_test.head(3)
predictions = best_model.predict(sample)
result = pd.DataFrame({"prediction": predictions}, index=sample.index)
display(pd.concat([sample.reset_index(drop=True), result.reset_index(drop=True)], axis=1))
assert len(predictions) == 3
assert set(predictions).issubset(set(y.unique()))

## Entregable

Incluye en tu ficha: `EXPERIMENT_NAME`, `BATCH_ID`, `BEST_RUN_ID`, la métrica de validación usada para elegir, la métrica de test y una captura del smoke test.
